In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import json
import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

import gpjax as gpx

In [ ]:
cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

In [ ]:
from admpe.utils.helper import get_plot_labels
from admpe.data_management import (
    list_excitation_results,
    provide_excitation_results_overview,
    SAFELY_EXCITABLE_SETS_ROOT,
)

from admpe.evaluation.excitation_evaluation import (
    load_excitation_comparison,
    load_corresponding_modeling_results,
)
from admpe.evaluation.excitation_visualization import (
    plot_excitation_feature_grid,
    compare_excitation_feature_grids,
    shared_density_levels,
)
from admpe.evaluation.modeling_evaluation import load_testing_sequences
from admpe.evaluation.shared import visualize_rollout_on_density

---

## transform old sets to new format:

In [ ]:
from dmpe.utils.sets.shared import (
    check_in_set,
    load_results,
    DiscretizedSet,
    SlicedSet,
    load_discretized_set,
    save_discretized_set,
)

In [ ]:
# S_xu_dict = {
#     "fluid_tank": load_discretized_set(SAFELY_EXCITABLE_SETS_ROOT / "fluid_tank_S_xu_666a769d-8c1b-4b.json"),
#     "pendulum": load_discretized_set(SAFELY_EXCITABLE_SETS_ROOT / "pendulum_S_xu_02430b86-ae0d-42.json"),
#     "cart_pole": load_discretized_set(SAFELY_EXCITABLE_SETS_ROOT / "cart_pole_S_xu_8744b5d5-30e6-4b.json"),
# }

S_xu_dict = {
    "pmsm": load_discretized_set(SAFELY_EXCITABLE_SETS_ROOT / "pmsm_S_xu_91f90457-e66e-48.json"),
}

In [ ]:
S_xu_dict

In [ ]:
def convert_dmpe_set(json_path, output_path=None):
    """
    Convert a DMPE JSON set to a compact NPZ archive.

    Validate Cartesian grid ordering before discarding the full grid.
    """
    json_path = pathlib.Path(json_path)

    with json_path.open() as stream:
        data = json.load(stream)

    shape = tuple(data["unflattened_shape"])
    dim = len(shape)

    mask = np.asarray(
        data.pop("mask"),
        dtype=np.bool_,
    ).reshape(shape)

    grid = np.asarray(
        data.pop("grid"),
        dtype=np.float32,
    ).reshape(shape + (dim,))

    axes = []
    for dimension in range(dim):
        axis_values = []
    
        number_of_coordinates = shape[dimension]
    
        for coordinate_index in range(number_of_coordinates):

            # set the indices to zero except for the coordiante index in the momentary dimension
            grid_position = [0] * dim
            grid_position[dimension] = coordinate_index
    
            # Read the complete coordinate vector at this grid position.
            point = grid[tuple(grid_position)]
    
            # Keep only the coordinate belonging to this dimension.
            coordinate_value = point[dimension]
    
            axis_values.append(coordinate_value)
    
        axis = np.array(axis_values, dtype=grid.dtype)
    
        # Check that every coordinate is finite.
        for coordinate_value in axis:
            if not np.isfinite(coordinate_value):
                raise ValueError(
                    f"Axis {dimension} contains a nonfinite coordinate."
                )
    
        # Check that coordinates are strictly increasing.
        for coordinate_index in range(1, len(axis)):
            previous_value = axis[coordinate_index - 1]
            current_value = axis[coordinate_index]
    
            if current_value <= previous_value:
                raise ValueError(
                    f"Axis {dimension} is not strictly increasing."
                )
    
        axes.append(axis)
    
    
    # Verify that every grid point agrees with the extracted axes.
    for grid_position in np.ndindex(shape):
        point = grid[grid_position]
    
        for dimension in range(dim):
            coordinate_index = grid_position[dimension]
    
            axis = axes[dimension]
            expected_coordinate = axis[coordinate_index]
            actual_coordinate = point[dimension]
    
            if actual_coordinate != expected_coordinate:
                raise ValueError(
                    "The grid does not match the extracted Cartesian axes."
                )

    if output_path is None:
        output_path = json_path.with_suffix(".npz")
    else:
        output_path = Path(output_path)

    if output_path.exists():
        raise FileExistsError(output_path)

    np.savez_compressed(
        output_path,
        format_version=np.int64(1),
        source_file=np.asarray(json_path.name),
        mask=mask,
        **{
            f"axis_{i}": axis
            for i, axis in enumerate(axes)
        },
    )

    return output_path

In [ ]:
# Convert each JSON file once.
for json_path in sorted(
    SAFELY_EXCITABLE_SETS_ROOT.glob("*_S_xu_*.json")
):
    npz_path = json_path.with_suffix(".npz")

    if not npz_path.exists():
        convert_dmpe_set(json_path)

## new format:

In [ ]:
from admpe.sets import SafelyExcitableSet

In [ ]:
SafelyExcitableSet.from_path(SAFELY_EXCITABLE_SETS_ROOT / "fluid_tank_S_xu_666a769d-8c1b-4b.npz")

In [ ]:
new_S_xu_dict = {
    "fluid_tank": SafelyExcitableSet.from_path(SAFELY_EXCITABLE_SETS_ROOT / "fluid_tank_S_xu_666a769d-8c1b-4b.npz"),
    "pendulum": SafelyExcitableSet.from_path(SAFELY_EXCITABLE_SETS_ROOT / "pendulum_S_xu_02430b86-ae0d-42.npz"),
    "cart_pole": SafelyExcitableSet.from_path(SAFELY_EXCITABLE_SETS_ROOT / "cart_pole_S_xu_8744b5d5-30e6-4b.npz"),
    "pmsm": SafelyExcitableSet.from_path(SAFELY_EXCITABLE_SETS_ROOT / "pmsm_S_xu_91f90457-e66e-48.npz")
}

In [ ]:
new_S_xu_dict

In [ ]:
new_S_xu_dict["pmsm"]

#### visualize:

In [ ]:
new_S_xu_dict["pmsm"].visualize(use_contourf=True)

In [ ]:
new_S_xu_dict["pmsm"].visualize(use_contourf=False)

#### checkout effect on evaluation points:

In [ ]:
from admpe.evaluation.modeling_evaluation import generate_default_prediction_validation_points
from dmpe.evaluation.plotting_utils import plot_feature_combinations

In [ ]:
evaluation_points = generate_default_prediction_validation_points(4, 15)

In [ ]:
plot_feature_combinations(evaluation_points);

In [ ]:
S_xu = new_S_xu_dict["pmsm"]

In [ ]:
filtered_points = evaluation_points[S_xu.contains(evaluation_points)]
plot_feature_combinations(filtered_points, mode="contourf");

In [ ]:
S_xu

In [ ]:
set_, path = SafelyExcitableSet.from_system_name("pendulum")
set_